# Notebook 3: Feature Visualization — PCA Maps & Qualitative Analysis

This notebook explores **what DINOv3 features look like** by:
1. Projecting high-dimensional patch features to RGB via PCA (reproducing Figures 4, 13, 17)
2. Visualizing cosine similarity maps between patches (reproducing Figure 3)
3. Comparing features across multiple resolutions (reproducing Figure 4)
4. Visualizing CLS token attention patterns

These visualizations are key evidence for DINOv3's main claim: **Gram anchoring produces
clean, semantically coherent dense features** that stay stable even at very high resolutions.

## 1. Setup

In [ ]:
import torch, os, sys, numpy as np, matplotlib.pyplot as plt
from PIL import Image

sys.path.insert(0, 'dinov3')
sys.path.insert(0, 'dinov3-segmentation-study')

from src.model_loading import load_backbone_with_intermediate_layers, load_backbone
from src.visualization import compute_pca_features

MEAN = np.array([0.485, 0.456, 0.406])
STD = np.array([0.229, 0.224, 0.225])

# Load ViT-L backbone for feature extraction
feature_extractor = load_backbone_with_intermediate_layers(
    'vitl16', weights_path='weights/dinov3_vitl16.pth',
    device='cuda', autocast_dtype=torch.float32
)
print('Backbone loaded successfully.')

In [ ]:
# Helper: load and preprocess an image at a given resolution
from torchvision.transforms import v2

def load_image(path_or_url, resize=512):
    """
    Load an image and return (PIL image for display, normalized tensor for model).
    
    Args:
        path_or_url: local file path or HTTP URL
        resize: target resolution (both H and W)
    
    Returns:
        img_pil: PIL Image resized for display
        tensor: (1, 3, resize, resize) normalized tensor on CUDA
    """
    if path_or_url.startswith('http'):
        import requests
        from io import BytesIO
        img = Image.open(BytesIO(requests.get(path_or_url).content)).convert('RGB')
    else:
        img = Image.open(path_or_url).convert('RGB')
    
    transform = v2.Compose([
        v2.ToImage(),
        v2.Resize((resize, resize), antialias=True),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])
    tensor = transform(img).unsqueeze(0).cuda()
    img_resized = img.resize((resize, resize))
    return img_resized, tensor

print('Helper function ready.')

## 2. PCA Feature Maps (Reproducing Figure 13)

We project the 1024-dimensional patch features to 3 dimensions using PCA,
then map to RGB. **Semantically similar regions should have similar colors.**

This is the key visualization that demonstrates DINOv3's advantage over
DINOv2, SigLIP 2, and PEspatial — DINOv3 features are sharper and more coherent.

In [ ]:
# COCO validation images (publicly accessible)
URLS = [
    'http://images.cocodataset.org/val2017/000000039769.jpg',   # Two cats on couch
    'http://images.cocodataset.org/val2017/000000397133.jpg',   # Kitchen scene
    'http://images.cocodataset.org/val2017/000000252219.jpg',   # Person on motorcycle
    'http://images.cocodataset.org/val2017/000000087038.jpg',   # Giraffe
]

fig, axes = plt.subplots(len(URLS), 2, figsize=(12, 6 * len(URLS)))

for i, url in enumerate(URLS):
    img_pil, img_tensor = load_image(url, resize=512)
    
    with torch.no_grad():
        features = feature_extractor(img_tensor)  # list of [(1, 1024, 32, 32)]
    
    # PCA projection: (1024, 32, 32) → (32, 32, 3) RGB
    pca_rgb = compute_pca_features(features[0].squeeze(0))
    
    # Upscale for display
    pca_upscaled = np.array(Image.fromarray(
        (pca_rgb * 255).astype(np.uint8)
    ).resize((512, 512), Image.BILINEAR))
    
    axes[i, 0].imshow(img_pil)
    axes[i, 0].set_title('Input Image')
    axes[i, 1].imshow(pca_upscaled)
    axes[i, 1].set_title('DINOv3 PCA Features')
    for ax in axes[i]:
        ax.axis('off')

plt.suptitle('PCA Visualization of DINOv3 ViT-L Dense Features', fontsize=14)
plt.tight_layout()
os.makedirs('results/visualizations', exist_ok=True)
plt.savefig('results/visualizations/pca_features.png', dpi=150)
plt.show()

## 3. Cosine Similarity Maps (Reproducing Figure 3)

Pick a reference patch (marked with a red cross) and compute its cosine similarity
to all other patches. Good features should show:
- **High similarity** to semantically related regions (e.g., other cat patches)
- **Low similarity** to unrelated regions (e.g., couch/background)

This is the visualization that reveals the **patch-level consistency** problem
that Gram anchoring (Section 4) was designed to solve.

In [ ]:
def cosine_similarity_map(features, ref_h, ref_w):
    """
    Compute cosine similarity between a reference patch and all other patches.
    
    Args:
        features: (C, H, W) patch feature tensor
        ref_h, ref_w: coordinates of the reference patch in the feature grid
    
    Returns:
        (H, W) numpy array with similarity values in [-1, 1]
    """
    C, H, W = features.shape
    ref_feat = features[:, ref_h, ref_w]                    # (C,)
    ref_feat = ref_feat / ref_feat.norm()                   # L2 normalize
    
    flat = features.reshape(C, -1)                          # (C, H*W)
    flat = flat / flat.norm(dim=0, keepdim=True)            # L2 normalize each patch
    
    sim = (ref_feat @ flat).reshape(H, W)                   # cosine similarity
    return sim.cpu().numpy()


# Load first image and extract features
img_pil, img_tensor = load_image(URLS[0], resize=512)
with torch.no_grad():
    features = feature_extractor(img_tensor)[0].squeeze(0)  # (1024, 32, 32)

# Pick 4 reference points in the feature grid
ref_points = [(8, 8), (8, 24), (24, 8), (16, 16)]

fig, axes = plt.subplots(1, len(ref_points) + 1, figsize=(5 * (len(ref_points) + 1), 5))
axes[0].imshow(img_pil)
axes[0].set_title('Input')
axes[0].axis('off')

for i, (rh, rw) in enumerate(ref_points):
    sim_map = cosine_similarity_map(features, rh, rw)
    # Normalize to [0, 255] and upscale
    sim_norm = ((sim_map - sim_map.min()) / (sim_map.max() - sim_map.min()) * 255).astype(np.uint8)
    sim_upscaled = np.array(Image.fromarray(sim_norm).resize((512, 512), Image.BILINEAR))
    
    axes[i + 1].imshow(img_pil, alpha=0.3)
    axes[i + 1].imshow(sim_upscaled, cmap='jet', alpha=0.7)
    # Mark reference point with red cross
    px, py = rw * 16, rh * 16  # Convert feature grid coords to pixel coords
    axes[i + 1].plot(px, py, 'r+', markersize=20, markeredgewidth=3)
    axes[i + 1].set_title(f'Sim to ({rh},{rw})')
    axes[i + 1].axis('off')

plt.suptitle('Cosine Similarity Maps — DINOv3 ViT-L', fontsize=14)
plt.tight_layout()
plt.savefig('results/visualizations/cosine_similarity.png', dpi=150)
plt.show()

## 4. Multi-Resolution Feature Consistency (Reproducing Figure 4)

DINOv3 uses **RoPE** (Rotary Position Embeddings) instead of learned absolute positions.
This means the model can process images at **any resolution** without adaptation.

Here we show that features remain semantically consistent as resolution increases
from 256×256 up to 1024×1024 — a key advantage over models with fixed positional embeddings.

In [ ]:
resolutions = [256, 512, 768, 1024]
url = URLS[0]

fig, axes = plt.subplots(2, len(resolutions), figsize=(5 * len(resolutions), 10))

for j, res in enumerate(resolutions):
    img_pil, img_tensor = load_image(url, resize=res)
    
    with torch.no_grad():
        features = feature_extractor(img_tensor)[0].squeeze(0)  # (C, H, W)
    
    C, H, W = features.shape
    pca_rgb = compute_pca_features(features)
    pca_up = np.array(Image.fromarray((pca_rgb * 255).astype(np.uint8)).resize((512, 512), Image.BILINEAR))
    
    axes[0, j].imshow(img_pil)
    axes[0, j].set_title(f'{res}×{res}')
    axes[0, j].axis('off')
    
    axes[1, j].imshow(pca_up)
    axes[1, j].set_title(f'PCA ({H}×{W} patches)')
    axes[1, j].axis('off')

plt.suptitle('DINOv3 Features at Multiple Resolutions (ViT-L/16) — Thanks to RoPE', fontsize=14)
plt.tight_layout()
plt.savefig('results/visualizations/multi_resolution.png', dpi=150)
plt.show()

print('Observation: Features remain semantically consistent across resolutions.')
print('Higher resolutions produce finer-grained feature maps while preserving semantics.')

## 5. CLS Token Attention Approximation

We approximate the CLS token's attention by computing cosine similarity
between the CLS token and all patch tokens. This reveals which patches
the model considers most "salient" or important for image-level understanding.

In [ ]:
# Load backbone with CLS token access
backbone = load_backbone('vitl16', weights_path='weights/dinov3_vitl16.pth')

img_pil, img_tensor = load_image(URLS[0], resize=512)

with torch.no_grad():
    # Extract features with CLS token
    output = backbone.get_intermediate_layers(
        img_tensor, n=[23], reshape=True, return_class_token=True
    )
    patch_feats, cls_token = output[0]  # (1, 1024, 32, 32), (1, 1024)

# CLS-to-patch cosine similarity
C = cls_token.shape[-1]
cls_norm = cls_token.squeeze() / cls_token.squeeze().norm()
patches_flat = patch_feats.squeeze().reshape(C, -1)
patches_norm = patches_flat / patches_flat.norm(dim=0, keepdim=True)
attn = (cls_norm @ patches_norm).reshape(32, 32).cpu().numpy()

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(img_pil)
axes[0].set_title('Input Image')
axes[0].axis('off')

attn_norm = ((attn - attn.min()) / (attn.max() - attn.min()) * 255).astype(np.uint8)
attn_up = np.array(Image.fromarray(attn_norm).resize((512, 512)))
axes[1].imshow(img_pil, alpha=0.4)
axes[1].imshow(attn_up, cmap='hot', alpha=0.6)
axes[1].set_title('CLS Token Attention (cosine approx.)')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('results/visualizations/cls_attention.png', dpi=150)
plt.show()

print('The CLS token primarily attends to salient foreground regions.')
print('This is useful for image-level tasks like classification and retrieval.')

## Key Observations

1. **PCA maps** show that DINOv3 features cleanly separate semantic regions without any supervision — each object gets a distinct color
2. **Cosine similarity maps** are well-localized — a patch on a cat has high similarity to other cat patches but not to the couch
3. **Multi-resolution features** remain consistent from 256px to 1024px thanks to RoPE — the model naturally handles any input size
4. The **CLS token** attends to salient foreground regions — useful for image-level tasks like retrieval (Section 6.2.2)
5. These clean feature maps are the direct result of **Gram anchoring** (Section 4) — without it, the cosine maps degrade severely during long training (see Figure 6 in the paper)